In [20]:
import pandas as pd
import numpy as np

In [21]:
# Definice cesty a načtení dat
file_path = r'C:\Users\Vít Dohnal\Downloads\ICOS_FR-Fon_FLUXNET_2019-2026_v1.3_r1\ICOS_FR-Fon_FLUXNET_FLUXMET_DD_2019-2026_v1.3_r1.csv'
sitename = ''

# Načtení dat s automatickým převedením FLUXNET chybějících hodnot na np.nan
df_raw = pd.read_csv(file_path, na_values=['-9999', '-9999.0'], parse_dates=['TIMESTAMP'])

In [ ]:
#Definice klimatologického pozadí CO2 (roční průměry v ppm z NOAA/Mauna Loa)
# Zahrnuje vaše sledované období 2008-2025 (poslední roky jsou extrapolované/předběžné)
global_co2_annual = {
    2008: 385.2, 2009: 386.8, 2010: 389.0, 2011: 390.5, 2012: 393.1,
    2013: 395.3, 2014: 397.2, 2015: 399.5, 2016: 402.9, 2017: 405.0,
    2018: 407.4, 2019: 411.4, 2020: 414.2, 2021: 416.4, 2022: 418.5,
    2023: 421.0, 2024: 423.5, 2025: 425.5
}

In [22]:
##  Vektorizované namapování CO2 podle roku
# Z data extrahujeme rok a přes map() připojíme odpovídající koncentraci
df_raw['year'] = df_raw['TIMESTAMP'].dt.year
co2_background = df_raw['year'].map(global_co2_annual)

# Logika pro sestavení finální CO2 řady
if 'CO2_F' in df_raw.columns:
    # Pokud máme kvalitní lokální gap-filled data, použijeme je, 
    # ale případné zbylé díry (NaN) vyplníme globálním pozadím
    co2_series = df_raw['CO2_F'].fillna(co2_background)
    print("Používám lokální CO2_F s případnou výplní globálním pozadím.")
else:
    # Jinak aplikujeme čistou křivku globálního pozadí
    co2_series = co2_background
    print("Lokální CO2_F chybí. Aplikována meziroční dynamická křivka pozadí (Mauna Loa).")

Upozornění: Lokální měření CO2 chybí. Aplikuji fixní hodnotu pozadí (400 ppm).


In [23]:
# Sestavení forcing dataframe pro rsofun
df_forcing = pd.DataFrame({
    'sitename': sitename,
    'date': df_raw['TIMESTAMP'],
    'temp': df_raw['TA_F'],                  
    'vpd': df_raw['VPD_F'] * 100,            
    'patm': df_raw['PA_F'] * 1000,           
    'ppfd': df_raw['SW_IN_F'] * 0.46 * 0.0864, 
    'rain': df_raw['P_F'],                   
    'snow': 0.0,                             
    'co2': co2_series                   
})

In [24]:
# Dodatečné ošetření chybějících hodnot (např. neúspěšně interpolované surové CO2)
# Nahrazení zbylých NaN v CO2 globálním průměrem
if isinstance(co2_series, pd.Series):
    df_forcing['co2'] = df_forcing['co2'].fillna(400.0)

# Filtrace dnů, kde chybí základní meteorologie (na těch model běžet nemůže)
df_forcing_clean = df_forcing.dropna(subset=['temp', 'vpd', 'ppfd', 'rain'])

output_path = f'rsofun_forcing_{sitename}.parquet'
df_forcing_clean.to_parquet(output_path, index=False)
print(f"\nUloženo {len(df_forcing_clean)} kompletních řádků do: {output_path}")


Uloženo 2791 kompletních řádků do: rsofun_forcing_.parquet
